In [1]:
## Load prerequisites libraries
import numpy as np
import pandas as pd
import warnings #Possibly not required
import json
import arrow #Possibly not required
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb as db
import config

In [2]:
# Check lims dsn connection to database before proceeding
def get_lims_connection(db_config):
    global odbcLimsConnection
    if db_config['dbType'] == "oracle":
        import oracledb
        oracledb.init_oracle_client(config.lib_dir)
        odbcLimsConnection = oracledb.connect(user=db_config['limsdb_username'], 
                                            password=db_config['limsdb_password'],
                                            dsn=db_config['limsdb_dsn'])
        odbcLimsConnection.arraysize = 5000
        
    elif db_config['dbType']== "postgres":
        import psycopg2
        odbcLimsConnection = psycopg2.connect('dbname=' + db_config['limsdb_database'] + ' user=' + db_config['limsdb_username'] + 
                                            ' password=' + db_config['limsdb_password'] + ' host=' + db_config['limsdb_hostname'])

    else:
        import pyodbc
        odbcLimsConnection = pyodbc.connect('DRIVER=' + db_config['limsdb_driver'] + ';DATABASE=' + db_config['limsdb_database'] +
                                        ';UID=' + db_config['limsdb_username'] + ';PWD=' + db_config['limsdb_password']+
                                        ';SERVER=' + db_config['limsdb_hostname'] + ';PORT=' + db_config['limsdb_port'] +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection



In [5]:
import json

def load_config(config_path):
    with open(config_path, 'r') as file:
        config = json.load(file)
    return config

In [43]:
def run_pipeline_for_database(db_config):
    print(f"Running pipeline for {db_config['customerName']} at {db_config['limsdb_hostname']}")
    # Here you would add your data engineering pipeline logic
    # This could involve connecting to the database, processing data, etc.
    get_lims_connection(db_config)
    rawDataFrame = pd.read_sql('''
                  select 
                 sample_number as table_key
                 ,'sample' as table_name
                 ,status
                 ,changed_on 
                 ,login_date
                 ,sampled_date
                 ,recd_date
                 ,date_started
                 ,date_completed
                 ,date_reviewed
                 --, prep_date
                 --,cast(released_on as date) as released_on
                 --,cast(expiration_date as date) as expiration_date
                 ,EXTRACT(EPOCH FROM (recd_date - login_date)) * 1000 AS login_to_recieve_ms
                 ,EXTRACT(EPOCH FROM (sampled_date - recd_date)) * 1000 AS recd_to_sample_ms
                 ,EXTRACT(EPOCH FROM (date_started - recd_date)) * 1000 AS recd_to_started_ms -- started might happen before recieved
                 

                 ,EXTRACT(EPOCH FROM (date_completed - recd_date)) * 1000 AS recd_to_completed_ms
                 ,EXTRACT(EPOCH FROM (date_completed - date_started)) * 1000 AS started_to_completed_ms
                 

                 ,EXTRACT(EPOCH FROM (date_reviewed - recd_date)) * 1000 AS recd_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (date_reviewed - date_started)) * 1000 AS started_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (date_reviewed - date_completed)) * 1000 AS completed_to_revewed_ms


                 --,EXTRACT(EPOCH FROM (released_on - recd_date)) * 1000 AS recd_to_released_ms   
                 --,EXTRACT(EPOCH FROM (released_on - date_started)) * 1000 AS started_to_released_ms   
                 --,EXTRACT(EPOCH FROM (released_on - date_completed)) * 1000 AS completed_to_released_ms   
                 --,EXTRACT(EPOCH FROM (released_on - date_reviewed)) * 1000 AS reviewed_to_released_ms   


                ,EXTRACT(EPOCH FROM (changed_on - login_date)) * 1000 AS login_to_changed_ms
                 ,EXTRACT(EPOCH FROM (changed_on - recd_date)) * 1000 AS recd_to_changed_ms
                 ,EXTRACT(EPOCH FROM (changed_on - date_started)) * 1000 AS started_to_changed_ms
                 ,EXTRACT(EPOCH FROM (changed_on - date_completed)) * 1000 AS completed_to_changed_ms
                ,EXTRACT(EPOCH FROM (changed_on - date_reviewed)) * 1000 AS reviewed_to_changed_ms
                --,EXTRACT(EPOCH FROM (changed_on - released_on )) * 1000 AS released_to_changed_ms
   
                from sample
                --where status = 'A'

                 union all

                 select 
                 test_number as table_key
                 ,'test' as table_name
                 ,t.status
                 ,t.changed_on 
                 ,s.login_date
                 ,s.sampled_date
                 ,s.recd_date
                 ,t.date_started
                 ,t.date_completed
                 ,t.date_reviewed
                 --, t.prep_date
                 --,cast(s.released_on as date) as released_on
                 --,cast(s.expiration_date as date) as expiration_date
                 ,EXTRACT(EPOCH FROM (s.recd_date - s.login_date)) * 1000 AS login_to_recieve_ms
                 ,EXTRACT(EPOCH FROM (s.sampled_date - s.recd_date)) * 1000 AS recd_to_sample_ms
                 ,EXTRACT(EPOCH FROM (t.date_started - s.recd_date)) * 1000 AS recd_to_started_ms -- started might happen before recieved
                 

                 ,EXTRACT(EPOCH FROM (t.date_completed - s.recd_date)) * 1000 AS recd_to_completed_ms
                 ,EXTRACT(EPOCH FROM (t.date_completed - t.date_started)) * 1000 AS started_to_completed_ms
                 

                 ,EXTRACT(EPOCH FROM (t.date_reviewed - s.recd_date)) * 1000 AS recd_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (t.date_reviewed - t.date_started)) * 1000 AS started_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (t.date_reviewed - t.date_completed)) * 1000 AS completed_to_revewed_ms


                 --,EXTRACT(EPOCH FROM (t.released_on - s.recd_date)) * 1000 AS recd_to_released_ms   
                 --,EXTRACT(EPOCH FROM (t.released_on - s.date_started)) * 1000 AS started_to_released_ms   
                 --,EXTRACT(EPOCH FROM (t.released_on - t.date_completed)) * 1000 AS completed_to_released_ms   
                 --,EXTRACT(EPOCH FROM (t.released_on - t.date_reviewed)) * 1000 AS reviewed_to_released_ms   


                ,EXTRACT(EPOCH FROM (t.changed_on - s.login_date)) * 1000 AS login_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - s.recd_date)) * 1000 AS recd_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - t.date_started)) * 1000 AS started_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - t.date_completed)) * 1000 AS completed_to_changed_ms
                ,EXTRACT(EPOCH FROM (t.changed_on - t.date_reviewed)) * 1000 AS reviewed_to_changed_ms
                --,EXTRACT(EPOCH FROM (t.changed_on - t.released_on )) * 1000 AS released_to_changed_ms
   
                from test as t
                 inner join sample as s on s.sample_number = t.sample_number 
                --where t.status = 'A'
                 

                 union all

                 select 
                 result_number as table_key
                 ,'result' as table_name
                 ,t.status
                 ,t.changed_on 
                 ,s.login_date
                 ,s.sampled_date
                 ,s.recd_date
                 ,s.date_started
                 ,t.entered_on as date_completed
                 ,t.date_reviewed
                 --, s.prep_date
                 --,cast(s.released_on as date) as released_on
                 --,cast(s.expiration_date as date) as expiration_date
                 ,EXTRACT(EPOCH FROM (s.recd_date - s.login_date)) * 1000 AS login_to_recieve_ms
                 ,EXTRACT(EPOCH FROM (s.sampled_date - s.recd_date)) * 1000 AS recd_to_sample_ms
                 ,EXTRACT(EPOCH FROM (s.date_started - s.recd_date)) * 1000 AS recd_to_started_ms -- started might happen before recieved
                 

                 ,EXTRACT(EPOCH FROM (t.entered_on - s.recd_date)) * 1000 AS recd_to_completed_ms
                 ,EXTRACT(EPOCH FROM (t.entered_on - s.date_started)) * 1000 AS started_to_completed_ms
                 

                 ,EXTRACT(EPOCH FROM (t.date_reviewed - s.recd_date)) * 1000 AS recd_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (t.date_reviewed - s.date_started)) * 1000 AS started_to_revewed_ms
                 ,EXTRACT(EPOCH FROM (t.date_reviewed - t.entered_on)) * 1000 AS completed_to_revewed_ms


                 --,EXTRACT(EPOCH FROM (s.released_on - s.recd_date)) * 1000 AS recd_to_released_ms   
                 --,EXTRACT(EPOCH FROM (s.released_on - s.date_started)) * 1000 AS started_to_released_ms   
                 --,EXTRACT(EPOCH FROM (s.released_on - t.entered_on)) * 1000 AS completed_to_released_ms   
                 --,EXTRACT(EPOCH FROM (s.released_on - t.date_reviewed)) * 1000 AS reviewed_to_released_ms   


                ,EXTRACT(EPOCH FROM (t.changed_on - s.login_date)) * 1000 AS login_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - s.recd_date)) * 1000 AS recd_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - s.date_started)) * 1000 AS started_to_changed_ms
                 ,EXTRACT(EPOCH FROM (t.changed_on - t.entered_on)) * 1000 AS completed_to_changed_ms
                ,EXTRACT(EPOCH FROM (t.changed_on - t.date_reviewed)) * 1000 AS reviewed_to_changed_ms
                --,EXTRACT(EPOCH FROM (t.changed_on - s.released_on )) * 1000 AS released_to_changed_ms
   
                from result as t
                 inner join sample as s on s.sample_number = t.sample_number 
                --where t.status = 'A'
                 
                 ''',con=odbcLimsConnection)
        # Tag with source DB
    
    customerName = db_config["customerName"]

    logData = db.sql(f'''
        select table_key
        ,lower(table_name) as TABLE_NAME
        ,max(case when login_date_json = '█' then null else login_date_json end) as login_date_audit_log 
        ,max(case when RECD_DATE_json = '█' then null else RECD_DATE_json end) as RECD_DATE_audit_log  
        ,max(case when DATE_STARTED_JSON = '█' then null else DATE_STARTED_JSON end) as DATE_STARTED_audit_log 
        ,max(case when date_completed_JSON = '█' then null else date_completed_JSON end) as date_completed_audit_log   
        ,max(case when date_reviewed_JSON = '█' then null else date_reviewed_JSON end) as date_reviewed_audit_log   
        from read_parquet('C:/MLAR_DATA/T0001/MLAR_RAW_{customerName}.parquet') 
        
        group by  table_key, table_name
        ''').to_df()
    
    print(len(rawDataFrame))
    print(len(logData))
    
    rawDataFrame = rawDataFrame.merge(logData,left_on=['table_key','table_name'] ,right_on=['TABLE_KEY','TABLE_NAME'],how='inner')
    
    print(len(rawDataFrame))
    
    rawDataFrame = rawDataFrame.drop(columns=['TABLE_KEY', 'TABLE_NAME'])

    rawDataFrame['login_date_compare'] = rawDataFrame['login_date_audit_log'] == rawDataFrame['login_date']
    rawDataFrame['recd_date_compare'] = rawDataFrame['RECD_DATE_audit_log'] == rawDataFrame['recd_date']
    rawDataFrame['DATE_STARTED_compare'] = rawDataFrame['DATE_STARTED_audit_log'] == rawDataFrame['date_started']
    rawDataFrame['date_completed_compare'] = rawDataFrame['date_completed_audit_log'] == rawDataFrame['date_completed']
    rawDataFrame['date_reviewed_compare'] = rawDataFrame['date_reviewed_audit_log'] == rawDataFrame['date_reviewed']
    
    rawDataFrame['SOURCE_DB'] = customerName

    # Convert to table for parquet conversion
    table = pa.Table.from_pandas(rawDataFrame)

    # Convert to parquet
    fileName = 'C:/MLAR_DATA/T0001/FS_turnaround_time_table/TIME_BASED_'+ db_config["customerName"] +'.parquet'
    pq.write_table(table,fileName)

   
    

    

In [44]:

def main():
    config = load_config("config_db.json")
    for db in config['databases']:
        run_pipeline_for_database(db)

if __name__ == "__main__":
    main()

Running pipeline for X5 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_11916\2403128231.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''


155437
183023
155437
Running pipeline for X8 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_11916\2403128231.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''


200935
184459
183435
Running pipeline for X7 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_11916\2403128231.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''


194219
221448
194219


In [6]:
## Create data point to check if the final data matches the audit data
import duckdb as db

import pprint as pp

pp.pprint(list(db.sql("select * from read_parquet('C:/MLAR_DATA/T0001/MLAR_RAW_X5.parquet') ").to_df().columns))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

['MLAR_TYPE',
 'AUDIT_ACTION',
 'AUDIT_TIMESTAMP',
 'AUDIT_TYPE',
 'AUDIT_COUNTER',
 'TRANS_ORDER',
 'TABLE_NAME',
 'TABLE_KEY',
 'REASON',
 'RECORD_SIGNED',
 'USER_NAME',
 'USER_ROLE',
 'SAMPLE_NUMBER',
 'T0083',
 'T0083_SEVERITY',
 'TEXT_ID_JSON',
 'STATUS_JSON',
 'OLD_STATUS_JSON',
 'IN_SPEC_JSON',
 'IN_CAL_JSON',
 'RE_SAMPLE_JSON',
 'ORIGINAL_SAMPLE_JSON',
 'ALIQUOT_JSON',
 'PARENT_SAMPLE_JSON',
 'PARENT_ALIQUOT_JSON',
 'LOGIN_DATE_JSON',
 'LOGIN_BY_JSON',
 'RECD_DATE_JSON',
 'RECEIVED_BY_JSON',
 'DATE_STARTED_JSON',
 'STARTED_JSON',
 'DATE_COMPLETED_JSON',
 'DATE_REVIEWED_JSON',
 'PRODUCT_VERSION_JSON',
 'PROJECT_JSON',
 'SAMPLE_TYPE_JSON',
 'TEMPLATE_JSON',
 'STANDARD_JSON',
 'LOT_JSON',
 'PARTIAL_SPEC_JSON',
 'PROTOCOL_LEVEL_JSON',
 'IN_CONTROL_JSON',
 'INVESTIGATED_JSON',
 'COMPOSITE_JSON',
 'PARENT_COMPOSITE_JSON',
 'HAS_FLAGS_JSON',
 'CHARGE_ENTRY_JSON',
 'ANALYSIS_JSON',
 'VERSION_JSON',
 'DATE_RECEIVED_JSON',
 'PREP_JSON',
 'PREP_DATE_JSON',
 'PREP_BY_JSON',
 'REVIEWER_JSON

In [ ]:
import pprint as pp

pp.pprint(list(dt.columns))

In [12]:

customerName = 'X5'

logData = db.sql(f'''
        select table_key
        ,table_name 
        ,max(case when login_date_json = '█' then null else login_date_json end) as login_date_audit_log 
        ,max(case when RECD_DATE_json = '█' then null else RECD_DATE_json end) as RECD_DATE_audit_log  
        ,max(case when DATE_STARTED_JSON = '█' then null else DATE_STARTED_JSON end) as DATE_STARTED_audit_log 
        ,max(case when date_completed_JSON = '█' then null else date_completed_JSON end) as date_completed_audit_log   
        ,max(case when date_reviewed_JSON = '█' then null else date_reviewed_JSON end) as date_reviewed_audit_log   
        , source_db
        from read_parquet('C:/MLAR_DATA/T0001/MLAR_RAW_{customerName}.parquet') 
        
        group by  table_key, table_name, source_db
        ''').to_df()

In [13]:
logData

,TABLE_KEY,TABLE_NAME,login_date_audit_log,RECD_DATE_audit_log,DATE_STARTED_audit_log,date_completed_audit_log,date_reviewed_audit_log,SOURCE_DB
0,195606,RESULT,None,None,None,None,2022-09-08 09:27:11,jaguar_prod
1,195597,RESULT,None,None,None,None,2022-02-23 09:47:22,jaguar_prod
2,195593,RESULT,None,None,None,None,2022-02-23 09:47:22,jaguar_prod
3,38075,TEST,None,None,2022-02-17 12:15:05,2022-04-26 14:28:35,2022-04-26 14:28:42,jaguar_prod
4,38141,TEST,None,None,2022-02-17 12:15:05,2022-02-23 09:31:23,2022-03-14 08:42:20,jaguar_prod
...,...,...,...,...,...,...,...,...
183018,348438,RESULT,None,None,None,None,None,jaguar_prod
183019,81052,TEST,None,None,None,None,None,jaguar_prod
183020,348457,RESULT,None,None,None,None,None,jaguar_prod
183021,81065,TEST,None,None,None,None,None,jaguar_prod


## Reviewing the Data to ensure it was processed as desired

### What are the data we are pulling 

- Verify to see that the right number of transactions were pulled
- Check to see how much missing data there is 
- Look at the summary of each calculated field

In [31]:
## Verify to see that the right number of transactions were pulled
import duckdb as db

db.sql("select SOURCE_DB, count(*) from read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/*.parquet') group by SOURCE_DB")


┌───────────┬──────────────┐
│ SOURCE_DB │ count_star() │
│   int32   │    int64     │
├──────────────────────────┤
│          0 rows          │
└──────────────────────────┘

In [32]:
db.sql("select SOURCE_DB, avg(login_to_recieve_ms) from read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/*.parquet') group by SOURCE_DB")



┌───────────┬──────────────────────────┐
│ SOURCE_DB │ avg(login_to_recieve_ms) │
│   int32   │          double          │
├──────────────────────────────────────┤
│                0 rows                │
└──────────────────────────────────────┘

In [20]:
import duckdb
con = duckdb.connect()

# Print schema of the Parquet file
schema = con.execute("DESCRIBE SELECT * FROM read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/*.parquet')").fetchall()
print(schema)



BinderException: Binder Error: table "read_parquet" has duplicate column name "TABLE_KEY"

In [19]:
### How to compare dates from sample to data in audit trail 
### 
import numpy as np



db.sql("select * from read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/TIME_BASED_*.parquet')")





BinderException: Binder Error: table "read_parquet" has duplicate column name "TABLE_KEY"